# nano Beatrix — sixteen bound pairs with the records' keys held at one length

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_rec16.ipynb)

**The question**: Turning all sixteen of the told head's rotation pairs by fixed phases of the key's characters at each end-marker keeps a million competing keys apart on most models (`colab/nano_norm16.ipynb`): at most 9 wrong looks in 4,608 questions, against 33 to 252 with eight bound pairs. Each remaining wrong look comes from another row's record whose key pairs run longer than the true record's; with every competitor's pair lengths set to the true record's, 17 of the 18 go. Holding the key's pairs at one length at every end-marker removed that kind of miss on eight pairs, but it also lengthened the index entries' keys, which the trained models keep short, and those then led the wrong looks. This notebook holds the length at the records' end-markers only:

| part | what | why |
|---|---|---|
| L16R s1-s6 | the sixteen-pair recipe with the key's sixteen bound pairs held at one length (2.18, the sixteen-pair models' median at the true record) at the records' end-markers only, before the turn; the index entries' and the questions' keys keep their own lengths; the query untouched (series `rec16`, the name carries `_krt16r218`) | every record's pair lengths equal every true record's by construction, and the index entries stay short |

A record's end-marker is found from the bytes before it only (a grid record's key follows DATA and its address; a framed record's key lies inside an open frame), so the hold never reads ahead. The comparison is the six sixteen-pair models (stored on the Hub, not trained again). Everything else is that recipe at 24,000 steps, seeds 1-6. Their capacity is read on CPU runtimes beside this one (`colab/nano_capacity_r16_lane1.ipynb`, `colab/nano_capacity_r16_lane2.ipynb`) as the models land.

**The cells**: C1 pins the trainer and the arm library and holds the decision block; C2 builds the data, checks the comparison models on the Hub, runs 200 steps of the first part on this card for the price line (a projection past 1.5x the plan stops for a re-plan) and probes the push path; C3 trains the six models one at a time and reads each one's look as it lands; C4 renders the tables. Every finished run and record ships to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/`, `bed/weights/<run>/` with the phases in `key_rot.safetensors`, `bed/reports/look/`).

**Before C1**: Runtime > Change runtime type > the G4 card. The notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` ends the session between runs; a restarted session resumes where it stopped. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/key_rotary.py`, `bed_train.py`, `rotary_scale.py`, `capacity.py`; `tests/test_records_hold.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.88" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 88), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, the looks
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run and record ships here; tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (the scale test's corpus)
BED = dict(                                            # the scale test's Alhena recipe at 24,000 steps, verbatim
    ctx=4096, mb=8, steps=24000,                       # 24,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,
)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows
                 head='digit', head_lam=1.0,           # the per-digit address head at the questions' SEPs (the scale test's term)
                 trace_every=2000,                     # the formation trace every 2,000 steps (the pointer's masses read at each point)
                 hard_read=1)                          # the index on 64 key-hard rows at the end
WIDE = dict(rotary_content=4)                          # ALHENA-W: 4 position / 4 content / 8 address pairs
VALUE = dict(value_addr=1)                             # ALHENA-V: the value-path tables, born zero (the name carries _va1)
POINTER = dict(ptr_lam=0.1, ptr_block=4, ptr_head=0)  # ALHENA-P: block 4 of 6, head 0, lambda .1 (the pointer screen's rule)
HARD = dict(grid_hard=0.5)                             # ALHENA-PK: the last 2 of the batch's 4 grid rows key-hard, from birth (the name carries _kh50)
TAIL = lambda L: f'_L{L}_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
ROT = dict(key_rot=1)                                  # the key in the told head's rotation
LARGER = dict(grid_comps=48)                           # the larger row: 48 records in 64-byte compartments, 24 asked (the name carries _gc48)
SIXTEEN = dict(key_pairs=16)                           # all sixteen of the told head's pairs bound at the coded end-markers (the name carries _krt16)
RECORDS = dict(key_norm=2, key_rho=2.18)               # the key's bound pairs held at length 2.18 at the records' end-markers only (the name carries _krt16r218)
TAG_L16R = lambda s: f'alhena-both_s{s}{TAIL(24000)}_cp4_gc48_kh50_va1_pt10b4h0_krt16r218_ahdigit1_rec16'
TAG_L16 = lambda s: f'alhena-both_s{s}{TAIL(24000)}_cp4_gc48_kh50_va1_pt10b4h0_krt16_ahdigit1_norm16'   # the comparison, stored
L16R = lambda s: (f'L16R s{s}', TAG_L16R(s), s, dict(BED, **GRID_ROWS, **HARD, **WIDE, **VALUE, **POINTER, **ROT, **LARGER, **SIXTEEN, **RECORDS, arms='alhena-both', seeds=str(s), series='rec16'))
PARTS = [L16R(s) for s in range(1, 7)]                 # the plan's order: L16R s1 .. s6
LOOK_HEAD = (4, 0)                                     # block 4 (numbered from 1), head 0: the head the term trains
READ_BATCHES, CALIB_BATCHES, HARD_BATCHES = 64, 32, 16 # 512 read rows, 256 calibration rows, 128 key-hard rows (8 rows a batch)
SMOKE_STEPS = 200                                      # C2's short run of the first part on this card for the price line; its record stays local
PLAN_MIN_24K = 17.7                                    # the planned minutes of one 24,000-step core on this card (the sixteen-pair models' measured pace)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
PARALLEL = 1                                           # ONE run at a time: one 9M craft at 8 x 4,096 saturates the card
print('decision block:', [p[0] for p in PARTS], '| records a row', [p[3]['grid_comps'] for p in PARTS], '| bound pairs', [p[3].get('key_pairs', 8) for p in PARTS], '| the hold', [p[3].get('key_norm', 0) for p in PARTS], '| its length', [p[3].get('key_rho') for p in PARTS], '| key-hard rows', [p[3].get('grid_hard', 0) for p in PARTS],
      '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of the first part on this card (the price line), the push probe. No part trains.
import json, os, shutil, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner
from geolip.nano_beatrix import rotary_scale as rs
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
# the comparison: the six sixteen-pair models and their capacity records on the Hub (never trained again here)
from huggingface_hub import HfApi as _Api
_hub = set(_Api().list_repo_files(REPO))
_tw = [s for s in range(1, 7) if not all(f in _hub for f in (f'bed/weights/{TAG_L16(s)}/model.safetensors', f'bed/weights/{TAG_L16(s)}/key_rot.safetensors',
       f'bed/reports/bed_{TAG_L16(s)}.json', f'bed/reports/capacity_tail/{TAG_L16(s)}_captail_r192_s1.json'))]
assert not _tw, ('the comparison models are not all on the Hub', _tw)
print('the comparison: L16 s1-s6 on the Hub (weights, phases, records, capacity records): 6 of 6')
todo_steps = 0
for label, tag, seed, spec in PARTS:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo_steps += (p['runs'] - p['done_before']) * spec['steps']
smoke = dict(PARTS[0][3], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2, hard_batches=1)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_24k = 24000 * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
PRICE_OK = per_24k <= STOP_FACTOR * PLAN_MIN_24K
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one 24,000-step run about {per_24k:.1f} min (planned "
      f"{PLAN_MIN_24K}); the {todo_steps:,} steps left about "
      f"{per_24k * todo_steps / 24000 / 60:.1f} h (+ a few minutes of end reads and the look read a run)"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_rec16.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': [p[0] for p in PARTS]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano rec16: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

def ship_file(path, dst_dir, ship):
    # one record to dst_dir on the Hub: staged alone in its own folder, so the upload carries only it
    st = f'{OUT}/stage/{os.path.basename(path)[:-5]}'
    os.makedirs(st, exist_ok=True)
    shutil.copy(path, st)
    ship.submit(st, dst_dir)

In [ ]:
# C3 (T) — THE TWELVE MODELS, one run at a time in the plan's order, resume-first (runner_done.txt in OUT; a look record in OUT/look/ is not
# read again); a STOP file in OUT starts no new run; every finished run ships to REPO as it lands, and its look read (the told head's
# request on 512 read rows and 128 key-hard rows) right after it, shipped to bed/reports/look/ (background shippers: the card never
# waits on the Hub). The time left is priced by the steps left at this session's measured pace.
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
lship = rs.Shipper(REPO) if push else None
done, t_all, steps_done = [], time.time(), 0
for n, (label, tag, seed, spec) in enumerate(PARTS, 1):
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f"=================== {label} ({n}/{len(PARTS)}, {spec['steps']:,} steps) ===================")
    t0 = time.time()
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((label, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', label, {k: v for k, v in res.items() if k != 'ran'})
        break
    assert os.path.exists(f'{OUT}/bed/{tag}/model.safetensors'), ('the run did not leave the expected folder', tag)
    out = f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'
    rec = rs.look_read(f'{OUT}/bed/{tag}', out, seed, HELD, look='ptr', read_batches=READ_BATCHES, hard_batches=HARD_BATCHES,
                       log=lambda s: None)
    assert (rec['look']['block'], rec['look']['head']) == LOOK_HEAD, (label, rec['look'])
    if lship:
        ship_file(out, 'bed/reports/look', lship)
    q, h = rec['read']['request'], rec['hard']['request']
    steps_done += spec['steps'] if res['ran'] else 0
    left = sum(p[3]['steps'] for p in PARTS[n:])
    pace = (time.time() - t_all) / max(1, steps_done)
    kru = (json.load(open(f'{OUT}/bed/bed_{tag}.json', encoding='utf-8'))['grid'].get('key_rot') or {}).get('use') or {}
    print(f"{label} ({spec.get('key_pairs', 8)} bound pairs, the records' keys held at {spec.get('key_rho')}): the key pairs score {kru.get('key_true', float('nan')):.2f} toward the true record, {kru.get('key_other', float('nan')):.2f} "
          f"toward the others; the look on the model's own row {q['LOOKF']['right']:.4f} (composed row {q['LOOK']['right']:.4f}; on the key-hard rows "
          f"{h['LOOKF']['right']:.4f}); the written index {q['INDEXF']['right']:.4f}; {(time.time() - t0) / 60:.1f} min; "
          f"{(time.time() - t_all) / 60:.1f} min spent; {len(PARTS) - n} models left ({left:,} steps), about {pace * left / 3600:.1f} h")
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED runs:', left if left else 'none')
if lship:
    print('look records shipped' if lship.drain() else 'look records still retrying in the background')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C4 — THE TABLES (the six models): the models (bits, the frame copy, the written index both ways, the key-hard rows both ways, the answers, the told head's
# mass at steps 2,000 / 4,000 / the end read, minutes), their looks (the requests' right, the look's no-request and wrong-record shares,
# the answers through each, the key-hard rows), and the key
# rotation (the told head's scores at the read rows' index entries by pair group, the key pairs' norms). Run any time.
import glob, json
from IPython.display import Markdown, display
rows = ['| run | bits | copy 32 / 128 / 256 / 512 | index TF / FR | key-hard rows TF / FR | answers true / via index | pointer mass 2k / 4k / read | min |',
        '|---|---|---|---|---|---|---|---|']
for label, tag, seed, spec in PARTS:
    p = f'{OUT}/bed/bed_{tag}.json'
    if not os.path.exists(p):
        continue
    r = json.load(open(p, encoding='utf-8'))
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    hd = g.get('hard')
    hard = f"{hd['index_right']:.4f} / {hd['indexf_right']:.4f}" if hd else '-'
    via = f"{g['true']['whole']:.4f} / {fr.get('indexf_whole', float('nan')):.4f}"
    pt_ = r['grid'].get('ptr') or {}
    tm = {t['step']: t.get('ptr_mass') for t in r['grid']['trace']}
    f_ = lambda v: '-' if v is None else f'{v:.3f}'
    ptr = f"{f_(tm.get(2000))} / {f_(tm.get(4000))} / {f_(pt_.get('mass_read'))}"
    rows.append(f"| {label} | {r['heldout_bpb']:.4f} | {copy} | {g['true']['index_right']:.4f} / {fr.get('indexf_right', float('nan')):.4f} | "
                f"{hard} | {via} | {ptr} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| model | written index (free-running) | look: composed / free-running / at the question | free-running look: none / wrong record | '
        'answers: true / written / look | look mass (decoded) / split / duplicate | key-hard rows: written / look | min |',
        '|---|---|---|---|---|---|---|---|']
for label, tag, seed, spec in PARTS:
    p = f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'
    if not os.path.exists(p):
        continue
    r = json.load(open(p, encoding='utf-8'))
    q, a, lk = r['read']['request'], r['read']['answers'], r['read']['look']['LOOKF']
    h = r.get('hard', {}).get('request', {})
    hard = f"{h['INDEXF']['right']:.4f} / {h['LOOKF']['right']:.4f}" if h else '-'
    rows.append(f"| {label} | {q['INDEXF']['right']:.4f} | {q['LOOK']['right']:.4f} / {q['LOOKF']['right']:.4f} / {q['LOOKQF']['right']:.4f} | "
                f"{lk['no_request']:.4f} / {lk['wrong_record']:.4f} | {a['LAYOUT']:.4f} / {a['INDEXF']:.4f} / {a['LOOKF']:.4f} | "
                f"{lk['decoded_mass_mean']:.3f} / {lk['split']:.4f} / {lk['duplicate']:.4f} | {hard} | {r['wall_s'] / 60:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| run | index entries | the key pairs: true record / others / itself | the address pairs: true / others | key-pair norm q / k at coded end-markers (elsewhere) |',
        '|---|---|---|---|---|']
for label, tag, seed, spec in PARTS:
    p = f'{OUT}/bed/bed_{tag}.json'
    if not os.path.exists(p):
        continue
    u = (json.load(open(p, encoding='utf-8'))['grid'].get('key_rot') or {}).get('use') or {}
    if not u:
        continue
    rows.append(f"| {label} | {u['entries']} | {u['key_true']:.2f} / {u['key_other']:.2f} / {u['key_self']:.2f} | {u['anchor_true']:.2f} / "
                f"{u['anchor_other']:.2f} | {u['q_key_norm_coded']:.2f} / {u['k_key_norm_coded']:.2f} ({u['q_key_norm_other']:.2f} / "
                f"{u['k_key_norm_other']:.2f}) |")
display(Markdown('\n'.join(rows)))